# 13 Implementing a Supervisor Node and Worker Agents

Notebook generado a partir del paquete Python. Los módulos se incluyen en orden de dependencia (los módulos importados por otros aparecen primero).

> Nota: los `import` de módulos locales del paquete original se conservan tal cual. Como todos los módulos están combinados en este notebook en orden de dependencia, los símbolos referenciados ya quedan definidos en celdas anteriores.

## ¿Qué hace este notebook?

Implementa la arquitectura **supervisor–trabajadores** (multi-agente). Un nodo
`supervisor` decide, mediante un LLM, qué trabajador actúa a continuación
(`researcher`, `analyst`, `writer`) o si el trabajo ha terminado (`FINISH`), enrutando con
`Command`. Cada trabajador es un agente con su propio *prompt* de rol.

El notebook combina, en orden de dependencia, el utilitario `llm_provider` (configura
`ChatAnthropic`), el módulo `supervisor_graph` (construye el grafo) y `main` (bucle
interactivo de tareas).

## Ejemplo de uso

**Datos de interacción que espera el agente.** No hay pausa humana: el `supervisor` decide
internamente qué trabajador actúa y cuándo `FINISH`. Concluye en un `invoke`.

- Entrada inicial esperada: `{"messages": [HumanMessage(content="<tarea>")]}`.
- El estado de salida acumula los mensajes de cada agente (cada uno con su `name`).

```python
from langchain_core.messages import HumanMessage

llm = get_llm()
app = build_graph(llm)
final_state = app.invoke(
    {"messages": [HumanMessage(content="Investiga y redacta un resumen sobre LangGraph")]}
)                                                   # supervisor enruta hasta FINISH
for msg in final_state["messages"]:
    role = msg.name if msg.name else "user/system"
    print(f"[{role.upper()}] {msg.content}\n")
```

In [1]:
import os
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic

In [2]:

# Load environment variables
load_dotenv()

True

In [3]:


def get_llm():
    # Read Anthropic API key
    api_key = os.getenv("ANTHROPIC_API_KEY")

    # Read model name
    model = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")

    # Validate API key presence
    if not api_key:
        raise ValueError("ANTHROPIC_API_KEY is missing in .env")

    # Return configured Claude LLM
    return ChatAnthropic(
        model=model,
        api_key=api_key,
        temperature=0,
    )

In [4]:
from __future__ import annotations
from typing import List

from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.types import Command

In [5]:


class State(MessagesState):
    """Shared state across supervisor and workers."""
    next: str  # Next node decision made by supervisor

In [6]:


def make_supervisor_node(llm, members: List[str]):
    # System prompt defining strict routing rules for the supervisor
    system_prompt = (
        "You are a supervisor managing a team of workers.\n"
        f"Workers: {members}\n\n"
        "STRICT RULES:\n"
        "- You MUST choose exactly one of the following options:\n"
        f"  {members + ['FINISH']}\n"
        "- Start with researcher.\n"
        "- Then analyst.\n"
        "- Then writer.\n"
        "- Choose FINISH ONLY after writer has responded.\n\n"
        "Return ONLY ONE WORD from the allowed options."
    )

    def supervisor(state: State) -> Command:
        # Combine system prompt with conversation history
        messages = [SystemMessage(content=system_prompt), *state["messages"]]
        response = llm.invoke(messages)

        # Normalize supervisor decision
        choice = response.content.strip().upper()

        # Fallback safety if model returns invalid choice
        if choice not in {m.upper() for m in members} | {"FINISH"}:
            choice = "RESEARCHER"

        # End graph execution
        if choice == "FINISH":
            return Command(goto=END)

        # Route to selected worker
        return Command(goto=choice.lower())

    return supervisor

In [7]:


def make_worker_node(llm, worker_name: str, role_prompt: str):
    def worker(state: State) -> Command:
        # Worker receives role instructions + full message history
        messages: List[BaseMessage] = [
            SystemMessage(content=role_prompt),
            *state["messages"],
        ]

        response = llm.invoke(messages)

        # Append worker output and return control to supervisor
        return Command(
            update={
                "messages": [
                    HumanMessage(content=response.content, name=worker_name)
                ]
            },
            goto="supervisor",
        )

    return worker

In [8]:


def build_graph(llm):
    # Ordered list of worker roles
    workers = ["researcher", "analyst", "writer"]

    graph = StateGraph(State)

    # Add supervisor node
    graph.add_node("supervisor", make_supervisor_node(llm, workers))

    # Add worker nodes
    graph.add_node(
        "researcher",
        make_worker_node(
            llm,
            "researcher",
            "Role: Researcher\nGather background information and key points.",
        ),
    )

    graph.add_node(
        "analyst",
        make_worker_node(
            llm,
            "analyst",
            "Role: Analyst\nAnalyze the problem and outline the approach.",
        ),
    )

    graph.add_node(
        "writer",
        make_worker_node(
            llm,
            "writer",
            "Role: Writer\nProduce the final clear response.",
        ),
    )

    # Entry point of the graph
    graph.add_edge(START, "supervisor")

    return graph.compile()

In [10]:
from langchain_core.messages import HumanMessage


In [11]:


def main():
    # Print application header
    print("\nSupervisor–Worker Architecture (LangGraph)\n")
    print("Enter a task. Type 'exit()' to quit.\n")

    # Initialize LLM and graph
    llm = get_llm()
    app = build_graph(llm)

    # Interactive task loop
    while True:
        user_request = input("Task: ").strip()

        # Exit condition
        if user_request.lower() in {"exit()", "exit", "quit"}:
            print("\nExiting.\n")
            return

        # Handle empty input
        if not user_request:
            print("Please enter a task.\n")
            continue

        # Invoke graph with user message
        final_state = app.invoke(
            {"messages": [HumanMessage(content=user_request)]}
        )

        print("\n========== FINAL OUTPUT ==========\n")

        # Print message trace
        for msg in final_state["messages"]:
            role = msg.name if msg.name else "user/system"
            print(f"[{role.upper()}]\n{msg.content}\n")

In [12]:


if __name__ == "__main__":
    main()


Supervisor–Worker Architecture (LangGraph)

Enter a task. Type 'exit()' to quit.



KeyboardInterrupt: 